# Data Cleaning

This notebook details data cleaning steps applied to the initial raw FSA data pull (as completed in `./01_data_exploration.ipynb`, saved in `../data/raw/fsa_london_establishments.json`).  

### Load raw data & flatten into DataFrame

The data is loaded from the saved JSON file - raw data pull save issues should surface here, so this also acts as a simple verification step.  


In [1]:
import json, pandas as pd

# load JSON and store in variable
with open("../data/raw/fsa_london_establishments.json") as f:
    all_establishments = json.load(f)

# check record count
print(f"Loaded {len(all_establishments)} records.")

# store in DataFrame
df = pd.json_normalize(all_establishments) # pd.json_normalize rather than pd.DataFrame
    # required due to nested fields (e.g. geocode -> long/lat, scores -> individual score values)
    # non-normalised DF would keep these nested fields as dict objects inside single cells
    # ...which is not especially helpful

# inspect stored DF
df.info()

Loaded 81217 records.
<class 'pandas.DataFrame'>
RangeIndex: 81217 entries, 0 to 81216
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   AddressLine1                   81217 non-null  str    
 1   AddressLine2                   81217 non-null  str    
 2   AddressLine3                   81217 non-null  str    
 3   AddressLine4                   81217 non-null  str    
 4   BusinessName                   81217 non-null  str    
 5   BusinessType                   81217 non-null  str    
 6   BusinessTypeID                 81217 non-null  int64  
 7   ChangesByServerID              81217 non-null  int64  
 8   Distance                       0 non-null      object 
 9   FHRSID                         81217 non-null  int64  
 10  LocalAuthorityBusinessID       81217 non-null  str    
 11  LocalAuthorityCode             81217 non-null  str    
 12  LocalAuthorityEmailAddress     8121

-> 81,217 row loaded matches the most recent raw pull: this is good news.   

Nested values have populated into the DataFrame as expected, e.g. `geocode.longitude`.  

In terms of null values found by column:
- `Distance` is a 'dead' column, completely empty because no search radius was provided in the API request.  
- `scores.` columns have approx. 70k non-null values, roughly aligning with the number of premises with a 'gradable' rating.  
- `geocode.longitude` and `geocode.latitude` have approx. 67k non-null values. The missing values are potentially mobile businesses - this will need some further investigation.  
- All other key columns are fully populated.  

### Investigate null values:

The null values have some likely potential explanations, as above, however in order to double-check the partially filled `scores.` and `geocode.` columns:

In [ ]:
# null nested 'scores' values check:

# get the null percentage of a group of values
def get_missing_percentage(group_data):
    return group_data.isna().mean()

# apply to nested Hygiene scores, grouping by RatingKey
df.groupby("RatingKey")["scores.Hygiene"].apply(get_missing_percentage)

RatingKey
fhrs_0_en-gb                      0.013333
fhrs_1_en-gb                      0.003580
fhrs_2_en-gb                      0.007011
fhrs_3_en-gb                      0.010556
fhrs_4_en-gb                      0.014971
fhrs_5_en-gb                      0.010927
fhrs_awaitinginspection_en-gb     1.000000
fhrs_awaitingpublication_en-gb    1.000000
fhrs_exempt_en-gb                 1.000000
Name: scores.Hygiene, dtype: float64

-> The awaiting/exempt ratings are fully empty - as expected.  

Numeric, 'gradable' ratings having approx. 1% of values missing was not expected - a small but non-zero number.  

In terms of the borough spread of missing nested score values:

In [ ]:
# filter to numeric rating, missing hygiene score:
missing_scores = df[df["RatingValue"].isin(["0","1","2","3","4","5"]) & df["scores.Hygiene"].isna()]

# output total and borough counts
print(len(missing_scores)) 
print(missing_scores["LocalAuthorityName"].value_counts())

805
LocalAuthorityName
Westminster                   172
Croydon                        82
Sutton                         57
Waltham Forest                 57
Barnet                         55
City of London Corporation     48
Kingston-Upon-Thames           48
Lambeth                        46
Enfield                        45
Wandsworth                     44
Hackney                        35
Southwark                      30
Hammersmith and Fulham         28
Richmond-Upon-Thames           20
Merton                         13
Haringey                        9
Kensington and Chelsea          9
Bexley                          3
Camden                          3
Hillingdon                      1
Name: count, dtype: int64


-> 805 total found

Spread is across 20 boroughs, rather than an input/administrative gap in a single borough.

In terms of the top-10 by missing percentage:

In [ ]:
# isolate gradable (numeric) ratings
gradable_df = df[df["RatingValue"].isin(["0","1","2","3","4","5"])]

# return Series for missing count (from above step) and total gradable count
missing_by_authority = missing_scores["LocalAuthorityName"].value_counts()
total_by_authority = gradable_df["LocalAuthorityName"].value_counts()

# Series / Series (aligns by LA Name index) to get pct, DESC order
missing_rate = (missing_by_authority / total_by_authority).sort_values(ascending=False)

# print top 10
print(missing_rate.head(10))

LocalAuthorityName
Sutton                        0.048635
Kingston-Upon-Thames          0.037915
Westminster                   0.032644
Waltham Forest                0.030695
Croydon                       0.029088
City of London Corporation    0.027923
Barnet                        0.023246
Enfield                       0.021226
Lambeth                       0.019159
Wandsworth                    0.018197
Name: count, dtype: float64


-> 805 gradable establishments (approx. 1.1%) are missing sub-scores. Missing rate varies by borough, with Sutton (4.9%) being the highest.  

This could be inconsistent Local Authority reporting, or could be driven by another factor.  

I'll also check `BusinessType`, in case there is a pattern:

In [ ]:
# extract missing/gradable counts:
missing_by_type = missing_scores["BusinessType"].value_counts()
total_by_type = gradable_df["BusinessType"].value_counts()

# assemble DF, aligning Series by LA Name label
business_type_summary = pd.DataFrame({
    "missing": missing_by_type,
    "total": total_by_type
})

# fillna(0) avoids NaN if a business category has no missing scores
business_type_summary["missing"] = business_type_summary["missing"].fillna(0) 

# add 'rate' column with percentage missing
business_type_summary["rate"] = business_type_summary["missing"] / business_type_summary["total"]

print(business_type_summary.sort_values("rate", ascending=False))

                                       missing  total      rate
BusinessType                                                   
Takeaway/sandwich shop                   173.0   8526  0.020291
Restaurant/Cafe/Canteen                  431.0  25175  0.017120
Hotel/bed & breakfast/guest house         13.0    943  0.013786
Pub/bar/nightclub                         36.0   3707  0.009711
Retailers - supermarkets/hypermarkets     15.0   2259  0.006640
Caring Premises                           27.0   4281  0.006307
School/college/university                 17.0   3226  0.005270
Other catering premises                   26.0   6019  0.004320
Retailers - other                         58.0  13519  0.004290
Mobile caterer                             7.0   1927  0.003633
Manufacturers/packers                      2.0    846  0.002364
Importers/Exporters                        0.0    123  0.000000
Distributors/Transporters                  0.0    428  0.000000
Farmers/growers                         

-> Takeaways/Sandwich Shops have the highest proportion at 2.0%, with Restaurant/Cafe/Canteen having the highest total (431).  

No dramatic outliers are seen from this approach, where business type clearly explains the missing values. It may be the case that some takeaway / restaurant type premises have simpler inspections where the establishments are particularly simple operations, or it may be something like a re-inspection focuses only on the final rating.    

At this point - given the relatively low totals and prevalence - I'm not too concerned by these missing values, despite the lack of a clear explanation. Where the nested `scores` values are needed in a model, these rows can be dropped.  